# Day 068 — Exercise 5: Document Reader Pipeline

**What you'll build:** `read_document(source, source_type, ocr_fn)` — the unified pipeline that dispatches to image OCR or PDF text extraction and returns structured output.

**Why it matters:** One function that handles any document source type is the clean interface your downstream pipeline needs. The dict output is the input for Day 69's structured extraction step — Pydantic schemas on top of OCR text.

In [ ]:
import io
import re
from PIL import Image, ImageEnhance

def preprocess_for_ocr(img):
    out = img.convert('L')
    out = ImageEnhance.Contrast(out).enhance(2.0)
    w, h = out.size
    if w < 1000:
        scale = 1000 / w
        out = out.resize((int(w * scale), int(h * scale)), Image.Resampling.LANCZOS)
    return out

def ocr_image(img, ocr_fn=None, lang='eng', config=''):
    if ocr_fn is not None:
        return ocr_fn(img)
    import pytesseract
    return pytesseract.image_to_string(img, lang=lang, config=config)

def extract_numbers(text):
    pattern = r'\b\d{1,3}(?:,\d{3})*(?:\.\d+)?|\b\d+(?:\.\d+)?\b'
    raw = re.findall(pattern, text)
    result = []
    for s in raw:
        try:
            result.append(float(s.replace(',', '')))
        except ValueError:
            pass
    return sorted(result)

def extract_pdf_text(pdf_bytes):
    import pypdf
    reader = pypdf.PdfReader(io.BytesIO(pdf_bytes))
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ''
        pages.append({'page': i, 'text': text, 'chars': len(text)})
    return pages

import io, struct

def _minimal_pdf(text: str) -> bytes:
    '''Generate a tiny single-page PDF with the given text (Helvetica 12pt).'''
    lines = text.splitlines()
    tf_lines = ''.join(f'({ln}) Tj T* ' for ln in lines)
    stream = (
        f'BT /F1 12 Tf 50 750 Td {tf_lines}ET'
    ).encode()
    slen = len(stream)

    objs = {
        1: b'<< /Type /Catalog /Pages 2 0 R >>',
        2: b'<< /Type /Pages /Kids [3 0 R] /Count 1 >>',
        3: b'<< /Type /Page /Parent 2 0 R /MediaBox [0 0 612 792] /Resources << /Font << /F1 << /Type /Font /Subtype /Type1 /BaseFont /Helvetica >> >> >> /Contents 4 0 R >>',
        4: f'<< /Length {slen} >>\nstream\n'.encode() + stream + b'\nendstream',
    }
    buf = io.BytesIO()
    buf.write(b'%PDF-1.4\n')
    offsets = {}
    for num, obj_bytes in objs.items():
        offsets[num] = buf.tell()
        buf.write(f'{num} 0 obj\n'.encode())
        buf.write(obj_bytes)
        buf.write(b'\nendobj\n')

    xref_pos = buf.tell()
    buf.write(b'xref\n')
    buf.write(f'0 {len(objs)+1}\n'.encode())
    buf.write(b'0000000000 65535 f \n')
    for num in range(1, len(objs)+1):
        buf.write(f'{offsets[num]:010d} 00000 n \n'.encode())
    buf.write(
        f'trailer << /Size {len(objs)+1} /Root 1 0 R >>\n'
        f'startxref\n{xref_pos}\n%%EOF\n'.encode()
    )
    return buf.getvalue()

_test_pdf = _minimal_pdf('Hello PDF\nLine two')

from PIL import ImageDraw
_receipt_img = Image.new('RGB', (250, 60), 'white')
_d = ImageDraw.Draw(_receipt_img)
_d.text((10, 10), 'Item $5.99', fill='black')
_d.text((10, 30), 'Total $7.50', fill='black')


## Task

Implement `read_document(source, source_type='image', ocr_fn=None) -> dict`:

- `'image'` path: `preprocess_for_ocr` → `ocr_image(ocr_fn=ocr_fn)` → `extract_numbers` → return `{text, numbers, chars}`
- `'pdf'` path: `extract_pdf_text` → return `{pages, total_chars, page_count}`
- Unknown `source_type` → raise `ValueError`

Helper functions are provided in the given cell. All checks use mocks.

## Your Implementation

In [ ]:
def read_document(source, source_type: str = 'image',
                  ocr_fn=None) -> dict:
    """Unified document reader: OCR images or extract PDF text.

    For 'image':
        1. preprocess_for_ocr(source)
        2. ocr_image(preprocessed, ocr_fn)
        3. extract_numbers(text)
        Return: {text: str, numbers: list[float], chars: int}

    For 'pdf':
        extract_pdf_text(source)
        Return: {pages: list[dict], total_chars: int, page_count: int}

    Raises:
        ValueError for unknown source_type
    """
    raise NotImplementedError


In [ ]:
def read_document(source, source_type: str = 'image',
                  ocr_fn=None) -> dict:
    if source_type == 'image':
        preprocessed = preprocess_for_ocr(source)
        text = ocr_image(preprocessed, ocr_fn=ocr_fn)
        numbers = extract_numbers(text)
        return {'text': text, 'numbers': numbers, 'chars': len(text)}
    if source_type == 'pdf':
        pages = extract_pdf_text(source)
        return {
            'pages':       pages,
            'total_chars': sum(p['chars'] for p in pages),
            'page_count':  len(pages),
        }
    raise ValueError(
        f"Unknown source_type: {source_type!r}. Use 'image' or 'pdf'."
    )


## Automated checks

In [ ]:
score, total = 0, 5
try:
    _mock_ocr = lambda img: 'Item $5.99\nTotal $7.50'

    # Image path
    result = read_document(_receipt_img, source_type='image', ocr_fn=_mock_ocr)
    assert isinstance(result, dict), f"Expected dict, got {type(result)}"
    assert 'text' in result and 'numbers' in result and 'chars' in result, (
        f"Missing keys: {set(result.keys())}")
    score += 1; print("\u2705 image result has text/numbers/chars keys")

    assert result['text'] == 'Item $5.99\nTotal $7.50'
    score += 1; print("\u2705 image text matches mock output")

    assert 5.99 in result['numbers'] and 7.5 in result['numbers'], (
        f"Expected 5.99 and 7.5 in numbers: {result['numbers']}")
    score += 1; print(f"\u2705 numbers extracted: {result['numbers']}")

    # PDF path
    pdf_result = read_document(_test_pdf, source_type='pdf')
    assert 'pages' in pdf_result and 'total_chars' in pdf_result and 'page_count' in pdf_result, (
        f"Missing keys: {set(pdf_result.keys())}")
    assert pdf_result['page_count'] == 1
    score += 1; print("\u2705 pdf result has pages/total_chars/page_count keys")

    # Unknown source_type raises ValueError
    raised = False
    try:
        read_document(_receipt_img, source_type='audio')
    except ValueError:
        raised = True
    assert raised, "Unknown source_type should raise ValueError"
    score += 1; print("\u2705 unknown source_type raises ValueError")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def read_document(source, source_type: str = 'image',
                  ocr_fn=None) -> dict:
    if source_type == 'image':
        preprocessed = preprocess_for_ocr(source)
        text = ocr_image(preprocessed, ocr_fn=ocr_fn)
        numbers = extract_numbers(text)
        return {'text': text, 'numbers': numbers, 'chars': len(text)}
    if source_type == 'pdf':
        pages = extract_pdf_text(source)
        return {
            'pages':       pages,
            'total_chars': sum(p['chars'] for p in pages),
            'page_count':  len(pages),
        }
    raise ValueError(
        f"Unknown source_type: {source_type!r}. Use 'image' or 'pdf'."
    )
```

**Why `chars: int` in the image result?** Downstream callers often want to know if the OCR returned any text at all — `chars == 0` means the image had no readable text and the result should not be passed to a parser. The same signal (`chars == 0`) is used in the PDF path at the page level to detect scanned pages that need an OCR fallback.

</details>